# Xay Dung & Chuan Hoa Bang Cau Noi: job_skills

Notebook nay tap trung xay dung bang cau noi quan he Nhieu-Nhieu (Many-to-Many Bridge Table) **`job_skills`** giua bang `jobs` va bang `skills`:
- Moi ban ghi dai dien cho mot ky nang duoc yeu cau hoac uu tien cho mot vi tri tuyen dung cu the.
- Ho tro da phuong phap trich xuat: tu the tag truc tiep (`tag_direct`), tu tu khoa tieu de (`title_keyword`), tu yeu cau cong viec (`requirement_rule`), va tu mo ta cong viec (`description_rule`).
- Tinh toan diem tin cay (`confidence`) tu 0.800 den 1.000 tuong ung voi tung nguon trich xuat.
- Phan loai ky nang bat buoc (`is_required = True`) va ky nang diem cong/uu tien (`is_required = False`).
- Luu tru doan trich bang chung thuc te (`evidence_text`) giup de dang kiem chung va giai thich du lieu.
- Kiem tra nghiem ngat toan ven khoa ngoai (FK) voi `jobs` va `skills`, dam bao tinh duy nhat cua khoa chinh phuc hop `(job_id, skill_id)`.

In [1]:
import os
import re
import json
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cau hinh hien thi pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Moi truong xu ly bang job_skills da san sang!')

Moi truong xu ly bang job_skills da san sang!


## 1. Nap Du Lieu Bronze va Danh Muc skills Chuan Hoa

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')
skills_nb_path = os.path.abspath('04_skills.ipynb')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    bronze_details = json.load(f)

# Nap danh muc skills tu notebook 04_skills.ipynb
with open(skills_nb_path, 'r', encoding='utf-8') as f:
    nb4 = json.load(f)
scope4 = {}
exec(''.join(nb4['cells'][3]['source']), scope4)
skills_def = scope4['skills_raw_definition']

skill_lookup = {}
for idx, s in enumerate(skills_def):
    s_id = idx + 1
    skill_lookup[s['name']] = s_id

print(f'Tong so tin tuyen dung nap vao: {len(bronze_listings):,} tin')
print(f'Tong so ky nang tham chieu tu bang skills: {len(skill_lookup):,} ky nang')

Tong so ky nang duoc dinh nghia trong danh muc: 76 ky nang
Tong so nhom chuyen mon (categories): 10 nhom
Tong so tin tuyen dung nap vao: 2,327 tin
Tong so ky nang tham chieu tu bang skills: 76 ky nang


## 2. Thiet Ke Pipeline Trich Xuat Ky Nang Da Tang (Multi-Layer Extraction Pipeline)

Thiết kế bộ đối sánh biểu thức chính quy (Regex Matchers) tối ưu, kèm theo logic xếp hạng mức độ ưu tiên:
1. `tag_direct` (Confidence: 1.000): Trích từ trường `skills_raw` của trang danh mục.
2. `title_keyword` (Confidence: 0.950): Trích từ chức danh công việc trong tiêu đề `title_raw`.
3. `requirement_rule` (Confidence: 0.900): Trích từ phần yêu cầu ứng viên `requirements_raw`.
4. `description_rule` (Confidence: 0.800): Trích từ phần mô tả trách nhiệm công việc `responsibilities_raw`.
5. Phân loại bắt buộc (`is_required`): Gán False nếu ngữ cảnh chứa 'ưu tiên', 'điểm cộng', 'preferred', 'nice to have'.

In [3]:
# Xay dung Regex Matchers chuyen biet cho 76 ky nang de tranh false positives
skill_matchers = []
for idx, s in enumerate(skills_def):
    skill_id = idx + 1
    name = s['name']
    
    if name == 'Java':
        pat = r'\bjava\b(?!script)'
    elif name == 'JavaScript':
        pat = r'\bjavascript\b|\bjs\b'
    elif name == 'TypeScript':
        pat = r'\btypescript\b|\bts\b'
    elif name == 'C++':
        pat = r'\bc\+\+\b|\bcpp\b'
    elif name == 'C#':
        pat = r'\bc#\b|\bcsharp\b'
    elif name == 'Golang':
        pat = r'\bgolang\b|\bgo\b'
    elif name == '.NET':
        pat = r'\b\.net\b|\basp\.net\b'
    elif name == 'Git':
        pat = r'\bgit\b(?!lab|hub)'
    elif name == 'React':
        pat = r'\breact\b(?! native)|\breactjs\b'
    elif name == 'Vue.js':
        pat = r'\bvue\b|\bvuejs\b|\bvue\.js\b'
    elif name == 'Node.js':
        pat = r'\bnode\b|\bnodejs\b|\bnode\.js\b'
    elif name == 'Next.js':
        pat = r'\bnextjs\b|\bnext\.js\b'
    elif name == 'Nuxt.js':
        pat = r'\bnuxtjs\b|\bnuxt\.js\b'
    elif name == 'Express.js':
        pat = r'\bexpressjs\b|\bexpress\.js\b'
    elif name == 'PostgreSQL':
        pat = r'\bpostgres\b|\bpostgresql\b'
    elif name == 'Microsoft SQL Server':
        pat = r'\bsql server\b|\bmssql\b'
    elif name == 'Oracle Database':
        pat = r'\boracle\b'
    elif name == 'Google Cloud (GCP)':
        pat = r'\bgcp\b|\bgoogle cloud\b'
    elif name == 'Kubernetes':
        pat = r'\bkubernetes\b|\bk8s\b'
    elif name == 'CI/CD':
        pat = r'\bci/cd\b|\bcicd\b'
    elif name == 'LLM (Generative AI)':
        pat = r'\bllm\b|\bgenerative ai\b|\brag\b|\blarge language\b'
    elif name == 'iOS Development':
        pat = r'\bios\b'
    elif name == 'Android Development':
        pat = r'\bandroid\b'
    elif name == 'Manual Testing':
        pat = r'\bmanual test\b|\bmanual testing\b'
    elif name == 'Automation Testing':
        pat = r'\bautomation test\b|\bautomation testing\b|\bauto test\b'
    elif name == 'HTML/CSS':
        pat = r'\bhtml\b|\bcss\b'
    elif name == 'Tailwind CSS':
        pat = r'\btailwind\b|\btailwindcss\b'
    elif name == 'SQL':
        pat = r'\bsql\b'
    else:
        pat = r'\b' + re.escape(name.lower()) + r'\b'
        
    skill_matchers.append({
        'skill_id': skill_id,
        'skill_name': name,
        'regex': re.compile(pat, re.IGNORECASE)
    })

def check_is_preferred(context_text):
    if not context_text:
        return False
    s = str(context_text).lower()
    pref_terms = ['ưu tiên', 'uu tien', 'lợi thế', 'loi the', 'plus', 'nice to have', 'preferred', 'điểm cộng']
    return any(term in s for term in pref_terms)

print('Bo matcher va ham nhan dien ngu canh da duoc khoi tao thanh cong!')

Bo matcher va ham nhan dien ngu canh da duoc khoi tao thanh cong!


## 3. Thuc Thi Trich Xuat & Khoi Tao Bang job_skills Hoan Chinh

In [4]:
job_skills_dict = {}  # (job_id, skill_id) -> record
now_iso = datetime.now().isoformat()

for idx in range(len(bronze_listings)):
    job_id = idx + 1
    l = bronze_listings[idx]
    d = bronze_details[idx]
    
    # Chuan bi cac nguon van ban
    raw_tags = l.get('skills_raw') or []
    if isinstance(raw_tags, str):
        raw_tags = [x.strip() for x in raw_tags.split(',') if x.strip()]
    tag_str = ' '.join(raw_tags)
    
    title_raw = str(l.get('title_raw', '') or '')
    req_raw = str(d.get('requirements_raw', '') or '')
    resp_raw = str(d.get('responsibilities_raw', '') or '')
    
    for sm in skill_matchers:
        s_id = sm['skill_id']
        reg = sm['regex']
        key = (job_id, s_id)
        
        # 1. Thu tu uu tien 1: tag_direct (Confidence: 1.000)
        m_tag = reg.search(tag_str)
        if m_tag:
            ev = f"Tag: {tag_str[max(0, m_tag.start()-10):min(len(tag_str), m_tag.end()+25)].strip()}"
            job_skills_dict[key] = {
                'job_id': job_id,
                'skill_id': s_id,
                'is_required': True,
                'confidence': 1.000,
                'extraction_method': 'tag_direct',
                'evidence_text': ev[:255],
                'created_at': now_iso
            }
            continue
            
        # 2. Thu tu uu tien 2: title_keyword (Confidence: 0.950)
        m_title = reg.search(title_raw)
        if m_title:
            ev = f"Tieu de: {title_raw[:120].strip()}"
            job_skills_dict[key] = {
                'job_id': job_id,
                'skill_id': s_id,
                'is_required': True,
                'confidence': 0.950,
                'extraction_method': 'title_keyword',
                'evidence_text': ev[:255],
                'created_at': now_iso
            }
            continue
            
        # 3. Thu tu uu tien 3: requirement_rule (Confidence: 0.900)
        m_req = reg.search(req_raw)
        if m_req:
            start_p = max(0, m_req.start() - 30)
            end_p = min(len(req_raw), m_req.end() + 60)
            snippet = req_raw[start_p:end_p].replace('\n', ' ').strip()
            is_pref = check_is_preferred(snippet)
            ev = f"Yeu cau: ...{snippet}..."
            job_skills_dict[key] = {
                'job_id': job_id,
                'skill_id': s_id,
                'is_required': not is_pref,
                'confidence': 0.900,
                'extraction_method': 'requirement_rule',
                'evidence_text': ev[:255],
                'created_at': now_iso
            }
            continue
            
        # 4. Thu tu uu tien 4: description_rule (Confidence: 0.800)
        m_resp = reg.search(resp_raw)
        if m_resp:
            start_p = max(0, m_resp.start() - 30)
            end_p = min(len(resp_raw), m_resp.end() + 60)
            snippet = resp_raw[start_p:end_p].replace('\n', ' ').strip()
            ev = f"Mo ta: ...{snippet}..."
            job_skills_dict[key] = {
                'job_id': job_id,
                'skill_id': s_id,
                'is_required': True,
                'confidence': 0.800,
                'extraction_method': 'description_rule',
                'evidence_text': ev[:255],
                'created_at': now_iso
            }
            continue

df_job_skills = pd.DataFrame(list(job_skills_dict.values()))

print('Khoi tao bang job_skills thanh cong:')
print(f'  - Tong so cap lien ket (job_id, skill_id): {len(df_job_skills):,}')
print(f'  - So luong tin co it nhat 1 ky nang: {df_job_skills["job_id"].nunique():,} / {len(bronze_listings)} ({df_job_skills["job_id"].nunique()/len(bronze_listings)*100:.1f}%)')
print(f'  - Tinh duy nhat cua Primary Key (job_id, skill_id): {len(df_job_skills) == len(df_job_skills.drop_duplicates(subset=["job_id", "skill_id"]))}')
print(f'  - Toan ven khoa ngoai job_id: {df_job_skills["job_id"].between(1, len(bronze_listings)).all()} (100% hop le)')
print(f'  - Toan ven khoa ngoai skill_id: {df_job_skills["skill_id"].between(1, len(skills_def)).all()} (100% hop le)')

Khoi tao bang job_skills thanh cong:
  - Tong so cap lien ket (job_id, skill_id): 4,938
  - So luong tin co it nhat 1 ky nang: 1,253 / 2327 (53.8%)
  - Tinh duy nhat cua Primary Key (job_id, skill_id): True
  - Toan ven khoa ngoai job_id: True (100% hop le)
  - Toan ven khoa ngoai skill_id: True (100% hop le)


## 4. Phan Tich Thong Ke, Danh Gia Chat Luong & Preview Bang job_skills

In [5]:
print('--- 1. CAU TRUC BANG job_skills (INFO) ---')
print(df_job_skills.info())

print('\n--- 2. PHAN BO THEO PHUONG PHAP TRICH XUAT (extraction_method) ---')
method_counts = df_job_skills['extraction_method'].value_counts()
method_percent = df_job_skills['extraction_method'].value_counts(normalize=True) * 100
df_method = pd.DataFrame({'So cap lien ket': method_counts, 'Ty le (%)': method_percent.round(2)})
print(df_method.to_string())

print('\n--- 3. PHAN BO TINH CHAT KY NANG (is_required) ---')
req_counts = df_job_skills['is_required'].value_counts()
req_percent = df_job_skills['is_required'].value_counts(normalize=True) * 100
df_req = pd.DataFrame({'So luong': req_counts, 'Ty le (%)': req_percent.round(2)})
print(df_req.to_string())

print('\n--- 4. TOP 15 KY NANG DUOC DEMANDE NHIEU NHAT (SO LUONG JOB CAN) ---')
id_to_name = {idx + 1: s['name'] for idx, s in enumerate(skills_def)}
id_to_cat = {idx + 1: s['cat'] for idx, s in enumerate(skills_def)}
top_skills = df_job_skills['skill_id'].value_counts().head(15)
df_top = pd.DataFrame({
    'skill_id': top_skills.index,
    'Ten ky nang': [id_to_name[i] for i in top_skills.index],
    'Nhom chuyen mon': [id_to_cat[i] for i in top_skills.index],
    'So luong job': top_skills.values,
    'Ty le (%)': (top_skills.values / len(bronze_listings) * 100).round(2)
})
print(df_top.to_string(index=False))

print('\n--- 5. PREVIEW 10 DONG BANG job_skills (CHUAN POSTGRESQL) ---')
preview_cols = ['job_id', 'skill_id', 'is_required', 'confidence', 'extraction_method', 'evidence_text']
print(df_job_skills[preview_cols].head(10).to_string(index=False))

--- 1. CAU TRUC BANG job_skills (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 4938 entries, 0 to 4937
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_id             4938 non-null   int64  
 1   skill_id           4938 non-null   int64  
 2   is_required        4938 non-null   bool   
 3   confidence         4938 non-null   float64
 4   extraction_method  4938 non-null   str    
 5   evidence_text      4938 non-null   str    
 6   created_at         4938 non-null   str    
dtypes: bool(1), float64(1), int64(2), str(3)
memory usage: 927.7 KB
None

--- 2. PHAN BO THEO PHUONG PHAP TRICH XUAT (extraction_method) ---
                   So cap lien ket  Ty le (%)
extraction_method                            
requirement_rule              3426      69.38
tag_direct                     952      19.28
description_rule               386       7.82
title_keyword                  174       3.52

--- 3. PHAN